## Crawling Data Detik.com

Pada bagian ini, kita akan berlatih melakukan proses pengambilan data (*web crawling*) dengan mengumpulkan judul-judul berita olahraga terkini dari situs **Detik Health**.

Untuk melakukan ekstraksi data ini, terdapat tiga *library* utama yang akan digunakan:
* **Requests:** Berfungsi untuk mengirimkan permintaan (HTTP Request) ke server website dan mengambil kode HTML mentah darinya.
* **BeautifulSoup:** Berfungsi untuk mengurai dan menyusun struktur HTML sehingga elemen-elemen tertentu (seperti teks judul dan link) dapat dipilah dengan mudah.
* **Pandas:** Berfungsi untuk mengorganisir hasil data yang telah diekstrak ke dalam bentuk tabel (*DataFrame*) yang terstruktur rapi.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Menentukan URL target (Detik Sport) dan Headers
url = 'https://health.detik.com/indeks'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

# 2. Mengambil konten HTML dari website
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Mencari semua blok artikel berita
articles = soup.find_all('article')
data_berita = []

# 4. Mengekstrak judul dan tautan dari masing-masing artikel
for article in articles:
    title_tag = article.find('h3')
    link_tag = article.find('a')
    
    if title_tag and link_tag:
        judul = title_tag.get_text(strip=True)
        tautan = link_tag['href']
        
        data_berita.append({
            'Judul Berita': judul,
            'Tautan': tautan
        })

# 5. Menampilkan hasil dalam bentuk tabel Pandas
df_berita = pd.DataFrame(data_berita)
df_berita.head(10) # Menampilkan 10 berita teratas

,Judul Berita,Tautan
0,"Penampakan Gumpalan Rambut 1,5 Kg yang Diangka...",https://health.detik.com/fotohealth/d-8656417/...
1,Ternyata Manusia Masih Bisa Hidup Tanpa 5 Orga...,https://health.detik.com/berita-detikhealth/d-...
2,"Viral Makan Jahe Mentah Biar Nggak Flu, Dokter...",https://health.detik.com/berita-detikhealth/d-...
3,"Wanita Texas Rayakan Ultah ke-105, Ungkap Raha...",https://health.detik.com/berita-detikhealth/d-...
4,"Jalan Cepat Vs Jumlah Langkah 10 Ribu Sehari, ...",https://health.detik.com/kebugaran/d-8656115/j...
5,Video: Pengakses Healing 119 Capai 600 per Shi...,https://20.detik.com/detikupdate/20260909-2609...
6,Video: Kemenkes soal Siswi di Karo Diduga Hila...,https://20.detik.com/detikupdate/20260909-2609...
7,Video Kemenkes: Kasus Bunuh Diri RI Naik Tiap ...,https://20.detik.com/detikupdate/20260909-2609...
8,Video Kemenkes Ungkap Masalah Keluarga Pemicu ...,https://health.detik.com/detiktv/d-8656064/vid...
9,"Waspada, 7 Makanan Ini Diam-diam Ganggu Keseha...",https://health.detik.com/berita-detikhealth/d-...


Data tabular di atas merupakan hasil akhir dari proses pengumpulan informasi mentah pada halaman indeks. Setelah dikumpulkan dalam format tabel seperti ini, dataset sudah siap untuk dibersihkan dan diproses lebih lanjut pada tahap prapemrosesan teks (*text preprocessing*) dalam siklus Web Mining.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

headers = {
    'User-Agent': 'Mozilla/5.0'
}

url = 'https://health.detik.com/indeks'

data_berita = []

# Maksimal 10 halaman
for halaman in range(10):

    print(f"Mengambil halaman {halaman + 1}...")

    response = requests.get(
        url,
        headers=headers,
        timeout=10
    )

    if response.status_code != 200:
        print("Gagal mengakses halaman")
        break

    soup = BeautifulSoup(response.text, 'html.parser')

    articles = soup.find_all('article')

    print("Artikel ditemukan:", len(articles))

    for article in articles:

        title_tag = article.find('h3')
        link_tag = article.find('a')

        if title_tag and link_tag:

            judul = title_tag.get_text(strip=True)
            tautan = urljoin(
                url,
                link_tag.get('href')
            )

            data_berita.append({
                'Judul Berita': judul,
                'Tautan': tautan
            })

        if len(data_berita) >= 200:
            break

    if len(data_berita) >= 200:
        break

    # Cari link Next
    next_link = soup.find(
        'a',
        string=lambda x: x and x.strip().lower() == 'next'
    )

    if next_link:
        url = urljoin(url, next_link.get('href'))
    else:
        print("Halaman berikutnya tidak ditemukan.")
        break


# Membuat DataFrame
df_berita = pd.DataFrame(data_berita)

# Hapus duplikat
df_berita = df_berita.drop_duplicates()

# Batasi 200 data
df_berita = df_berita.head(200)

# Nomor urut
df_berita.index = range(1, len(df_berita) + 1)
df_berita.index.name = 'No'

print("\nJumlah berita:", len(df_berita))

display(df_berita)

Mengambil halaman 1...
Artikel ditemukan: 20
Mengambil halaman 2...
Artikel ditemukan: 20
Mengambil halaman 3...
Artikel ditemukan: 20
Mengambil halaman 4...
Artikel ditemukan: 20
Mengambil halaman 5...
Artikel ditemukan: 20
Mengambil halaman 6...
Artikel ditemukan: 20
Mengambil halaman 7...
Artikel ditemukan: 20
Mengambil halaman 8...
Artikel ditemukan: 20
Mengambil halaman 9...
Artikel ditemukan: 20
Mengambil halaman 10...
Artikel ditemukan: 20

Jumlah berita: 200


,Judul Berita,Tautan
No,,
1,"Terungkap Lewat Studi, Kebiasaan Minum Panas T...",https://health.detik.com/berita-detikhealth/d-...
2,"Penampakan Gumpalan Rambut 1,5 Kg yang Diangka...",https://health.detik.com/fotohealth/d-8656417/...
3,Ternyata Manusia Masih Bisa Hidup Tanpa 5 Orga...,https://health.detik.com/berita-detikhealth/d-...
4,"Viral Makan Jahe Mentah Biar Nggak Flu, Dokter...",https://health.detik.com/berita-detikhealth/d-...
5,"Wanita Texas Rayakan Ultah ke-105, Ungkap Raha...",https://health.detik.com/berita-detikhealth/d-...
...,...,...
196,Menkes Respons Usulan IDI soal Gaji Dokter Rp ...,https://health.detik.com/berita-detikhealth/d-...
197,"Heboh 'Tambal Gigi Online' Berujung Infeksi, K...",https://health.detik.com/berita-detikhealth/d-...
198,Infografis: 7 Kota Paling Stres di Asia Tengga...,https://health.detik.com/infografis/d-8644202/...


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

headers = {
    'User-Agent': 'Mozilla/5.0'
}

url = 'https://health.detik.com/indeks'


data_berita = []


for halaman in range(10):

    print(f"Mengambil halaman {halaman + 1}...")

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        response.raise_for_status()

    except requests.exceptions.RequestException as e:
        print("Terjadi kesalahan:", e)
        break

    # Parsing HTML
    soup = BeautifulSoup(response.text, 'html.parser')

    # Mencari semua artikel
    articles = soup.find_all('article')

    print("Artikel ditemukan:", len(articles))

    # Mengambil judul dan tautan
    for article in articles:

        title_tag = article.find('h3')
        link_tag = article.find('a')

        if title_tag and link_tag:

            judul = title_tag.get_text(strip=True)
            tautan = link_tag.get('href')

            # Jika tautan kosong, lanjutkan
            if not tautan:
                continue

            # Mengubah tautan menjadi URL lengkap
            tautan = urljoin(url, tautan)

            # Menambahkan data berita
            data_berita.append({
                'Judul Berita': judul,
                'Tautan': tautan
            })

        # Berhenti jika sudah mendapatkan 200 berita
        if len(data_berita) >= 200:
            break

    # Jika sudah mendapatkan 200 data
    if len(data_berita) >= 200:
        break

    # =========================
    # MENCARI LINK HALAMAN BERIKUTNYA
    # =========================

    next_link = None

    for link in soup.find_all('a'):

        teks_link = link.get_text(strip=True).lower()

        if teks_link == 'next':
            next_link = link.get('href')
            break

    # Jika tidak ada halaman berikutnya
    if not next_link:
        print("Halaman berikutnya tidak ditemukan.")
        break

    # Pindah ke halaman berikutnya
    url = urljoin(url, next_link)

df_berita = pd.DataFrame(data_berita)

# Menghapus data duplikat
df_berita = df_berita.drop_duplicates()

# Membatasi maksimal 200 data
df_berita = df_berita.head(200)

# Membuat nomor urut
df_berita.index = range(1, len(df_berita) + 1)
df_berita.index.name = 'No'


print("\nJumlah data berita:", len(df_berita))

print("\n========== 10 DATA BERITA PERTAMA ==========")
display(df_berita.head(10))

print("\n========== 10 DATA BERITA TERAKHIR ==========")
display(df_berita.tail(10))


# Menggabungkan semua judul berita
semua_judul = ' '.join(df_berita['Judul Berita'])

# Mengubah menjadi huruf kecil dan mengambil kata
kata = re.findall(r'\b\w+\b', semua_judul.lower())

# Menghilangkan angka
kata = [k for k in kata if not k.isdigit()]

# Mengambil semua kata unik
kata_unik = list(set(kata))

# Mengurutkan kata unik agar rapi
kata_unik.sort()

df_kata_unik = pd.DataFrame(
    kata_unik,
    columns=['Kata Unik']
)

# Membuat nomor urut
df_kata_unik.index = range(1, len(df_kata_unik) + 1)
df_kata_unik.index.name = 'No'

print("\nJumlah seluruh kata unik:", len(df_kata_unik))

print("\n========== 10 KATA UNIK PERTAMA ==========")
display(df_kata_unik.head(10))

print("\n========== 10 KATA UNIK TERAKHIR ==========")
display(df_kata_unik.tail(10))

Mengambil halaman 1...
Artikel ditemukan: 20
Mengambil halaman 2...
Artikel ditemukan: 20
Mengambil halaman 3...
Artikel ditemukan: 20
Mengambil halaman 4...
Artikel ditemukan: 20
Mengambil halaman 5...
Artikel ditemukan: 20
Mengambil halaman 6...
Artikel ditemukan: 20
Mengambil halaman 7...
Artikel ditemukan: 20
Mengambil halaman 8...
Artikel ditemukan: 20
Mengambil halaman 9...
Artikel ditemukan: 20
Mengambil halaman 10...
Artikel ditemukan: 20

Jumlah data berita: 199

========== 10 DATA BERITA PERTAMA ==========


,Judul Berita,Tautan
No,,
1,"Minum Kopi Memang Menyehatkan, Tapi Kalau Kelebihan Bisa Begini Dampaknya ke Tulang",https://health.detik.com/berita-detikhealth/d-8656594/minum-kopi-memang-menyehatkan-tapi-kalau-kelebihan-bisa-begini-dampaknya-ke-tulang
2,"Terungkap Lewat Studi, Kebiasaan Minum Panas Tingkatkan Risiko Kanker Kerongkongan",https://health.detik.com/berita-detikhealth/d-8656494/terungkap-lewat-studi-kebiasaan-minum-panas-tingkatkan-risiko-kanker-kerongkongan
3,"Penampakan Gumpalan Rambut 1,5 Kg yang Diangkat dari Perut Gadis 16 Tahun",https://health.detik.com/fotohealth/d-8656417/penampakan-gumpalan-rambut-1-5-kg-yang-diangkat-dari-perut-gadis-16-tahun
4,Ternyata Manusia Masih Bisa Hidup Tanpa 5 Organ Ini,https://health.detik.com/berita-detikhealth/d-8656357/ternyata-manusia-masih-bisa-hidup-tanpa-5-organ-ini
5,"Viral Makan Jahe Mentah Biar Nggak Flu, Dokter Gizi Ungkap Faktanya",https://health.detik.com/berita-detikhealth/d-8656124/viral-makan-jahe-mentah-biar-nggak-flu-dokter-gizi-ungkap-faktanya
6,"Wanita Texas Rayakan Ultah ke-105, Ungkap Rahasia Panjang Umurnya",https://health.detik.com/berita-detikhealth/d-8656121/wanita-texas-rayakan-ultah-ke-105-ungkap-rahasia-panjang-umurnya
7,"Jalan Cepat Vs Jumlah Langkah 10 Ribu Sehari, Mana yang Lebih Baik?",https://health.detik.com/kebugaran/d-8656115/jalan-cepat-vs-jumlah-langkah-10-ribu-sehari-mana-yang-lebih-baik
8,"Video: Pengakses Healing 119 Capai 600 per Shift, Ada Anak Usia 8-10 Tahun",https://20.detik.com/detikupdate/20260909-260909068/video-pengakses-healing-119-capai-600-per-shift-ada-anak-usia-8-10-tahun
9,Video: Kemenkes soal Siswi di Karo Diduga Hilang Ingatan Usai Keracunan MBG,https://20.detik.com/detikupdate/20260909-260909066/video-kemenkes-soal-siswi-di-karo-diduga-hilang-ingatan-usai-keracunan-mbg



========== 10 DATA BERITA TERAKHIR ==========


,Judul Berita,Tautan
No,,
190,"Video Bukan Cuma ISPA, Asap Karhutla Bisa Picu Asma hingga PPOK",https://health.detik.com/detiktv/d-8644517/video-bukan-cuma-ispa-asap-karhutla-bisa-picu-asma-hingga-ppok
191,"Viral Kasus Dugaan Pelecehan Dokter Obgyn Via DM, POGI Angkat Bicara",https://health.detik.com/berita-detikhealth/d-8644484/viral-kasus-dugaan-pelecehan-dokter-obgyn-via-dm-pogi-angkat-bicara
192,"Suka Tidur Pakai Makeup Selama 20 Tahun, Wanita Ini Alami Robekan Kornea Berulang",https://health.detik.com/true-story/d-8644214/suka-tidur-pakai-makeup-selama-20-tahun-wanita-ini-alami-robekan-kornea-berulang
193,"Video Karhutla Picu 50.891 Kasus ISPA Juli-Agustus, 12.600 di Antaranya Balita",https://20.detik.com/detikupdate/20260902-260902109/video-karhutla-picu-50891-kasus-ispa-juli-agustus-12600-di-antaranya-balita
194,Daftar 10 Buah Terbaik untuk Kesehatan Jantung Menurut Ahli Kardiologi,https://health.detik.com/berita-detikhealth/d-8644435/daftar-10-buah-terbaik-untuk-kesehatan-jantung-menurut-ahli-kardiologi
195,"6 Kesalahan Diam-diam Bisa Bikin Gagal Diet, Nomor 4 Paling Sering Dialami",https://health.detik.com/diet/d-8644213/6-kesalahan-diam-diam-bisa-bikin-gagal-diet-nomor-4-paling-sering-dialami
196,"Menkes Respons Usulan IDI soal Gaji Dokter Rp 30-110 Juta, Bahas Bareng Kemenkeuâ",https://health.detik.com/berita-detikhealth/d-8644403/menkes-respons-usulan-idi-soal-gaji-dokter-rp-30-110-juta-bahas-bareng-kemenkeu
197,"Heboh 'Tambal Gigi Online' Berujung Infeksi, Kok Bisa Sih Ada yang Beli?",https://health.detik.com/berita-detikhealth/d-8643953/heboh-tambal-gigi-online-berujung-infeksi-kok-bisa-sih-ada-yang-beli
198,"Infografis: 7 Kota Paling Stres di Asia Tenggara, Ada DKI Jakarta",https://health.detik.com/infografis/d-8644202/infografis-7-kota-paling-stres-di-asia-tenggara-ada-dki-jakarta



Jumlah seluruh kata unik: 916

========== 10 KATA UNIK PERTAMA ==========


,Kata Unik
No,
1,a
2,abu
3,ac
4,ada
5,agar
6,agustus
7,ahli
8,ahlinya
9,ai



========== 10 KATA UNIK TERAKHIR ==========


,Kata Unik
No,
907,warganya
908,waspada
909,waspadai
910,water
911,wellous
912,wilayah
913,yamal
914,yang
915,yotania
